In [ ]:
print("hello world")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm

# Enable progress_apply for pandas
tqdm.pandas()


def harmonize_orca_answer(orca_text: str) -> str:
    """Extracts the final number and appends the GSM8K delimiter format."""
    numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", str(orca_text))
    if numbers:
        final_number = numbers[-1].replace(",", "")
        if final_number.endswith("."):
            final_number = final_number[:-1]
        return f"{orca_text}\n#### {final_number}"
    return orca_text


def run_orcamath_eda(
    model_id: str = "meta-llama/Meta-Llama-3.1-8B-Instruct", sample_size: int = 15000
):
    print(f"1. Loading Dataset and Tokenizer (Sampling {sample_size} rows)...")
    # Orca Math only has a train split
    dataset = load_dataset("microsoft/orca-math-word-problems-200k", split="train")

    # Shuffle and select subset to keep EDA fast but representative
    train_df = dataset.shuffle(seed=42).select(range(sample_size)).to_pandas()

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    print("\n2. Applying Output Harmonization...")
    # train_df["harmonized_answer"] = train_df["answer"].apply(harmonize_orca_answer)
    train_df["harmonized_answer"] = train_df["answer"]

    # Verify how many extractions failed (missing the delimiter)
    missing_delimiter = train_df[~train_df["harmonized_answer"].str.contains("####")]
    extraction_failure_rate = (len(missing_delimiter) / len(train_df)) * 100
    print(
        f"Rows missing '####' delimiter: {len(missing_delimiter)} ({extraction_failure_rate:.2f}% failure rate)"
    )

    print("\n3. Calculating Token Length Distributions...")

    def calculate_lengths(row):
        messages = [
            {
                "role": "system",
                "content": "You are a mathematical reasoning assistant. Solve the problem step-by-step and provide the final answer after ####.",
            },
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["harmonized_answer"]},
        ]

        formatted_string = tokenizer.apply_chat_template(messages, tokenize=False)
        prompt_only = messages[:2]
        formatted_prompt = tokenizer.apply_chat_template(prompt_only, tokenize=False)

        total_tokens = len(tokenizer.encode(formatted_string))
        prompt_tokens = len(tokenizer.encode(formatted_prompt))
        completion_tokens = total_tokens - prompt_tokens

        return pd.Series([prompt_tokens, completion_tokens, total_tokens])

    # Using progress_apply because tokenizer.encode on 15k rows takes a moment
    train_df[["prompt_tokens", "completion_tokens", "total_tokens"]] = (
        train_df.progress_apply(calculate_lengths, axis=1)
    )

    print("\nToken Length Percentiles:")
    print(
        train_df[["prompt_tokens", "completion_tokens", "total_tokens"]].describe(
            percentiles=[0.50, 0.90, 0.95, 0.99]
        )
    )

    print("\n4. Plotting Distributions...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.histplot(
        train_df["prompt_tokens"], bins=50, ax=axes[0], color="blue"
    ).set_title("Prompt Token Length")
    sns.histplot(
        train_df["completion_tokens"], bins=50, ax=axes[1], color="orange"
    ).set_title("Completion Token Length")
    sns.histplot(
        train_df["total_tokens"], bins=50, ax=axes[2], color="green"
    ).set_title("Total Sequence Length")
    plt.tight_layout()
    plt.show()

    print("\n5. Vocabulary Frequency Analysis (Router Prep)...")
    vectorizer = CountVectorizer(stop_words="english", max_features=20)
    word_counts = vectorizer.fit_transform(train_df["question"]).sum(axis=0)
    words_freq = [
        (word, word_counts[0, idx]) for word, idx in vectorizer.vocabulary_.items()
    ]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    words, counts = zip(*words_freq)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(counts), y=list(words), palette="viridis")
    plt.title("Top 20 Informative Words in Orca Math Questions")
    plt.show()


if __name__ == "__main__":
    run_orcamath_eda()


In [ ]:
import re
from datasets import load_dataset


def harmonize_orca_answer(orca_text: str) -> str:
    """Extracts the final number and appends the GSM8K delimiter format."""
    # Regex to find numbers, including negatives, commas, and decimals
    numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", str(orca_text))
    if numbers:
        final_number = numbers[-1].replace(",", "")
        if final_number.endswith("."):
            final_number = final_number[:-1]
        return f"{orca_text}\n#### {final_number}"
    return orca_text


def verify_harmonization(num_samples: int = 100):
    print(f"Loading {num_samples} random samples from Orca Math...")
    # Load dataset
    dataset = load_dataset("microsoft/orca-math-word-problems-200k", split="train")

    # Shuffle with a seed to get a consistent random assortment of problem types
    sample_data = dataset.shuffle(seed=42).select(range(num_samples))

    print(f"\n{'=' * 80}")
    print(f" DISPLAYING {num_samples} HARMONIZED EXAMPLES")
    print(f"{'=' * 80}\n")

    failure_count = 0

    for i, example in enumerate(sample_data):
        question = example["question"]
        original_answer = example["answer"]
        harmonized_answer = harmonize_orca_answer(original_answer)

        # Check if our regex successfully appended the delimiter
        if "####" not in harmonized_answer:
            failure_count += 1

        print(f"--- EXAMPLE {i + 1} ---")
        print(f"QUESTION:\n{question}\n")
        print(f"HARMONIZED ANSWER:\n{harmonized_answer}")
        print(f"\n{'=' * 80}\n")

    print(f"Verification Complete!")
    print(f"Total Extractions Failed: {failure_count} out of {num_samples}")

    if failure_count > 0:
        print("Note: Extractions fail if the text does not contain any numbers.")


if __name__ == "__main__":
    verify_harmonization(100)
